# Social Media → Donation Referrals: Causal Pipeline (v2)
### IS 455 — Machine Learning | CRISP-DM Framework

**Business Problem:** Which content decisions — post type, media format, call to action, sentiment tone, etc. — are most strongly associated with generating donation referrals on each platform?

**Who cares:** Social media managers and organizational leadership who create posts without a dedicated marketing team. They need actionable, platform-specific guidance: *"When I post on Instagram, what should I choose?"*

**Approach:** Explanatory (causal) OLS regression — one model per platform. SelectKBest feature selection applied to address overfitting diagnosed via cross-validation. A two-tier confidence system distinguishes platforms with sufficient data (Facebook, Instagram) from those with limited data (all others).

> **Predictive vs. Explanatory:** This pipeline is explicitly **explanatory**. We want to know *which content features are associated with more referrals, and by how much* — not to forecast future post performance. OLS coefficients and p-values are the primary outputs, not RMSE.

---
## Phase 1 — Business Understanding

### Feature Design
All features represent decisions the poster controls **before** publishing:

| Feature | Type | Decision |
|---|---|---|
| `post_type` | Categorical | Content format (ImpactStory, Campaign, etc.) |
| `media_type` | Categorical | Photo / Video / Reel / Carousel / Text |
| `has_call_to_action` | Boolean | Whether to include a CTA |
| `call_to_action_type` | Categorical | DonateNow / LearnMore / ShareStory / SignUp |
| `content_topic` | Categorical | Subject matter of the post |
| `sentiment_tone` | Categorical | Emotional register |
| `features_resident_story` | Boolean | Whether to feature an anonymized story |
| `is_boosted` | Boolean | Whether to pay for promotion |
| `campaign_name` | Categorical | Campaign association |
| `num_hashtags` | Numeric | Hashtag count |
| `day_of_week` | Categorical | Posting day |
| `post_hour` | Numeric | Posting time |
| `caption_length` | Numeric | Caption length |
| `follower_count_at_post` | Numeric | **Control variable** — partials out audience size |

### Excluded Features (mediators — sit between content decisions and referrals)
`impressions`, `reach`, `likes`, `comments`, `shares`, `saves`, `click_throughs`, `engagement_rate`, `profile_visits`, `video_views`, `watch_time_seconds`, `forwards`, `boost_budget_php`

---
## Phase 2 — Data Understanding

In [1]:
import warnings
import json
import os
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import SelectKBest, f_regression
import joblib

warnings.filterwarnings("ignore")
matplotlib.use("Agg")

# ── Data source config ───────────────────────────────────────────────────────
USE_DB = True

# CSV fallback — same Lighthouse bundle as reintegration_timeline_pipeline.ipynb
DEFAULT_DATA_DIR = Path.home() / "Downloads" / "lighthouse_csv_v7"
DATA_DIR = Path(os.environ.get("LIGHTHOUSE_CSV_DIR", str(DEFAULT_DATA_DIR))).expanduser().resolve()
# Optional full path to CSV; if empty, CSV mode uses DATA_DIR / "social_media_posts.csv"
SOCIAL_MEDIA_CSV_PATH = os.environ.get("SOCIAL_MEDIA_CSV_PATH", "").strip()


def _to_snake(name: str) -> str:
    import re

    name = name.replace(" ", "_").replace("-", "_")
    s1 = re.sub("(.)([A-Z][a-z]+)", r"\1_\2", name)
    s2 = re.sub("([a-z0-9])([A-Z])", r"\1_\2", s1)
    s2 = re.sub(r"__+", "_", s2)
    return s2.strip("_").lower()


def _normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [_to_snake(c) for c in df.columns]
    return df


def _find_dotenv(start: Optional[Path] = None) -> Optional[Path]:
    here = (start or Path.cwd()).resolve()
    for p in [here, *here.parents]:
        candidate = p / ".env"
        if candidate.exists():
            return candidate
    return None


def _make_sqlalchemy_url(db_override: Optional[str] = None) -> str:
    host = os.getenv('PGHOST')
    user = os.getenv('PGUSER')
    port = os.getenv('PGPORT', '5432')
    db   = db_override or os.getenv('PGDATABASE')
    pwd  = os.getenv('PGPASSWORD')
    if not all([host, user, db, pwd]):
        missing = [k for k in ['PGHOST','PGUSER','PGDATABASE','PGPASSWORD'] if not os.getenv(k)]
        raise ValueError(f'Missing required env vars (expected in .env): {missing}')
    return f'postgresql+psycopg2://{user}:{pwd}@{host}:{port}/{db}?sslmode=require'


def _pip_install(packages):
    import sys
    import subprocess

    cmd = [sys.executable, "-m", "pip", "install", "--user", *packages]
    print("Installing missing packages:", " ".join(packages))
    subprocess.check_call(cmd)


def _make_engine():
    try:
        from sqlalchemy import create_engine
    except ImportError:
        _pip_install(["sqlalchemy", "psycopg2-binary"])
        from sqlalchemy import create_engine

    try:
        from dotenv import load_dotenv
    except ImportError:
        _pip_install(["python-dotenv"])
        from dotenv import load_dotenv

    env_path = _find_dotenv()
    if env_path is None:
        raise FileNotFoundError("No .env found (expected in project folder or parents).")

    load_dotenv(env_path, override=False)
    url = _make_sqlalchemy_url(db_override=os.getenv("PGDATABASE_OVERRIDE"))

    # statement_timeout prevents long-hanging COUNTs/queries on managed PG
    return create_engine(url, pool_pre_ping=True, connect_args={"options": "-c statement_timeout=20000"})


def _read_table(engine, logical_name: str) -> pd.DataFrame:
    """Load a ORM-style table; columns are snake_cased to match CSV exports."""
    schema = os.getenv("PGSCHEMA", "public")
    TABLES = {
        "social_media_posts": "SocialMediaPosts",
    }
    table = os.getenv("SOCIAL_MEDIA_TABLE", TABLES[logical_name])
    df = pd.read_sql_query(f'SELECT * FROM "{schema}"."{table}"', con=engine)
    return _normalize_columns(df)


def _to_utc_naive(s: pd.Series) -> pd.Series:
    """Convert mixed tz-aware/naive datetimes to tz-naive UTC."""
    dt = pd.to_datetime(s, errors="coerce", utc=True)
    return dt.dt.tz_convert(None)


# ── Config ────────────────────────────────────────────────────────────────
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

OUTPUT_DIR = Path("model_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PLATFORMS = ["Facebook", "Instagram", "Twitter", "TikTok", "LinkedIn", "WhatsApp", "YouTube"]
LOW_N_THRESHOLD = 100

HIGH_CONFIDENCE_PLATFORMS = {"Facebook", "Instagram"}
HIGH_CONF_K = 10
LOW_CONF_K = 10

print("Libraries loaded ✓")
print(f"  USE_DB={USE_DB}")
print(f"  DATA_DIR={DATA_DIR}")
print(f"  DB table (override SOCIAL_MEDIA_TABLE): {os.getenv('SOCIAL_MEDIA_TABLE', 'SocialMediaPosts')}")

Libraries loaded ✓
  USE_DB=True
  DATA_DIR=/Users/jamescorrigan/Downloads/lighthouse_csv_v7
  DB table (override SOCIAL_MEDIA_TABLE): SocialMediaPosts


In [2]:
# ── Load posts: PostgreSQL (same .env as reintegration) or CSV fallback ─────
if USE_DB:
    engine = _make_engine()
    df_raw = _read_table(engine, "social_media_posts")
else:
    if SOCIAL_MEDIA_CSV_PATH:
        csv_path = Path(SOCIAL_MEDIA_CSV_PATH).expanduser().resolve()
    else:
        if not DATA_DIR.exists():
            raise FileNotFoundError(
                f"DATA_DIR not found: {DATA_DIR}. Set LIGHTHOUSE_CSV_DIR or SOCIAL_MEDIA_CSV_PATH."
            )
        csv_path = DATA_DIR / "social_media_posts.csv"
    if not csv_path.exists():
        raise FileNotFoundError(f"CSV not found: {csv_path}")
    df_raw = pd.read_csv(csv_path)
    df_raw = _normalize_columns(df_raw)

if "created_at" in df_raw.columns:
    df_raw["created_at"] = _to_utc_naive(df_raw["created_at"])

print(f"Shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
df_raw.head(3)

Shape: 812 rows × 39 columns


,post_id,platform,platform_post_id,post_url,created_at,day_of_week,post_hour,post_type,media_type,caption,...,video_views,engagement_rate,profile_visits,donation_referrals,estimated_donation_value_php,follower_count_at_post,watch_time_seconds,avg_view_duration_seconds,subscriber_count_at_post,forwards
0,318,WhatsApp,wa_4293211912553134,https://whatsapp.com/channel/lighthouse_ph/429...,2023-01-05 18:52:00,Thursday,18,FundraisingAppeal,Text,"This is hard to ask, but our reserve is gone. ...",...,NaN,0.1105,21,10,21473.25,1522,NaN,NaN,NaN,50.0
1,529,Instagram,ig_5129900136072862,https://instagram.com/p/sYhZp-0AvhH,2023-01-06 11:30:00,Friday,11,EducationalContent,Photo,What does freedom mean to a trafficking surviv...,...,NaN,0.1745,335,2,4708.45,1833,NaN,NaN,NaN,NaN
2,86,LinkedIn,li_2326736034499294,https://linkedin.com/feed/update/urn:li:activi...,2023-01-08 10:14:00,Sunday,10,EventPromotion,Text,SAVE THE DATE! Join us on January 21 for Fundr...,...,NaN,0.1411,8,0,0.00,457,NaN,NaN,NaN,NaN


In [3]:
# ── Platform counts ─────────────────────────────────────────────────────────
platform_counts = df_raw['platform'].value_counts()
print('Posts per platform:')
print(platform_counts.to_string())
print()
low_n = platform_counts[platform_counts < LOW_N_THRESHOLD]
if not low_n.empty:
    print(f'⚠️  LOW SAMPLE WARNING: {list(low_n.index)} have fewer than {LOW_N_THRESHOLD} posts.')
    print('   Models will be built but interpreted with reduced confidence.')

Posts per platform:
platform
Facebook     199
Instagram    164
Twitter      117
WhatsApp      93
TikTok        89
LinkedIn      79
YouTube       71

⚠️  LOW SAMPLE WARNING: ['WhatsApp', 'TikTok', 'LinkedIn', 'YouTube'] have fewer than 100 posts.
   Models will be built but interpreted with reduced confidence.


In [4]:
# ── Target distribution ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df_raw['donation_referrals'], bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Donation Referrals — Full Distribution')
axes[0].set_xlabel('Donation Referrals')
axes[0].set_ylabel('Count')
axes[1].hist(df_raw['donation_referrals'].clip(upper=80), bins=40, color='steelblue', edgecolor='white')
axes[1].set_title('Donation Referrals — Clipped at 80 (detail)')
axes[1].set_xlabel('Donation Referrals')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/target_distribution.png', dpi=120)
plt.show()

print(df_raw['donation_referrals'].describe())
zero_pct = (df_raw['donation_referrals'] == 0).mean() * 100
print(f'\nZero-referral posts: {zero_pct:.1f}%')
print('NOTE: Right-skewed with ~36% zeros. OLS is used for interpretability;')
print('Poisson/NB regression would fit the distribution better statistically.')

count    812.000000
mean      12.795567
std       31.261714
min        0.000000
25%        0.000000
50%        2.000000
75%       11.000000
max      458.000000
Name: donation_referrals, dtype: float64

Zero-referral posts: 35.7%
NOTE: Right-skewed with ~36% zeros. OLS is used for interpretability;
Poisson/NB regression would fit the distribution better statistically.


In [5]:
# ── Per-platform referral stats ──────────────────────────────────────────────
platform_stats = df_raw.groupby('platform')['donation_referrals'].agg(
    n='count', mean='mean', median='median', std='std', max='max'
).round(2)
print('Per-platform referral statistics:')
print(platform_stats.to_string())

Per-platform referral statistics:
             n   mean  median    std  max
platform                                 
Facebook   199  10.93     1.0  26.41  203
Instagram  164  11.58     2.0  22.80  145
LinkedIn    79   4.28     0.0  13.00   85
TikTok      89  19.55     4.0  35.97  239
Twitter    117   5.74     2.0  11.30   80
WhatsApp    93  23.10     2.0  56.70  458
YouTube     71  19.96     4.0  38.18  206


In [6]:
# ── Numeric feature correlations with target ────────────────────────────────
num_cols = ['num_hashtags', 'post_hour', 'caption_length', 'follower_count_at_post']
num_cols = [c for c in num_cols if c in df_raw.columns]
corr = df_raw[num_cols + ['donation_referrals']].corr()['donation_referrals'].drop('donation_referrals')
print('Pearson correlation with donation_referrals (numeric features):')
print(corr.sort_values(ascending=False).to_string())

Pearson correlation with donation_referrals (numeric features):
caption_length            0.137771
post_hour                 0.122168
follower_count_at_post   -0.010359
num_hashtags             -0.027421


---
## Phase 3 — Data Preparation

In [7]:
def prepare_platform_data(df: pd.DataFrame, platform: str):
    """
    Filter to one platform, one-hot encode categoricals, standardize numerics.
    Returns the FULL feature matrix before feature selection.
    """
    pdf = df[df['platform'] == platform].copy()
    n   = len(pdf)
    low = n < LOW_N_THRESHOLD

    y = pdf['donation_referrals'].astype(float).reset_index(drop=True)

    # Impute known-structural nulls (only if column exists — DB schema may differ from CSV)
    if 'call_to_action_type' in pdf.columns:
        pdf['call_to_action_type'] = pdf['call_to_action_type'].fillna('None')
    if 'campaign_name' in pdf.columns:
        pdf['campaign_name'] = pdf['campaign_name'].fillna('NoCampaign')

    cat_features  = ['post_type', 'media_type', 'call_to_action_type',
                     'content_topic', 'sentiment_tone', 'campaign_name', 'day_of_week']
    bool_features = ['has_call_to_action', 'features_resident_story', 'is_boosted']
    num_features  = ['num_hashtags', 'post_hour', 'caption_length', 'follower_count_at_post']

    cat_features = [c for c in cat_features if c in pdf.columns]
    bool_features = [c for c in bool_features if c in pdf.columns]
    num_features = [c for c in num_features if c in pdf.columns]

    # Drop single-value categoricals (no contrast possible)
    cat_features = [c for c in cat_features if pdf[c].nunique() > 1]

    ref_cats      = {}
    encoded_parts = []
    for col in cat_features:
        ref = pdf[col].value_counts().idxmax()   # most common = reference
        ref_cats[col] = ref
        dummies = pd.get_dummies(pdf[col], prefix=col, drop_first=False)
        if f'{col}_{ref}' in dummies.columns:
            dummies = dummies.drop(columns=[f'{col}_{ref}'])
        encoded_parts.append(dummies)

    bool_df = pdf[bool_features].astype(int).reset_index(drop=True)

    num_df = pdf[num_features].copy().reset_index(drop=True)
    for col in num_features:
        mu, sigma = num_df[col].mean(), num_df[col].std()
        if sigma > 0:
            num_df[col] = (num_df[col] - mu) / sigma

    X_full = pd.concat(
        [p.reset_index(drop=True) for p in encoded_parts] + [bool_df, num_df],
        axis=1
    ).astype(float)

    return X_full, y, n, low, ref_cats, cat_features


def select_features(X_full: pd.DataFrame, y: pd.Series, k: int):
    """
    Use SelectKBest (F-statistic) to reduce to top-k features.
    Returns the reduced DataFrame and the fitted selector.
    """
    k = min(k, X_full.shape[1])
    selector = SelectKBest(f_regression, k=k)
    selector.fit(X_full, y)
    selected_cols = X_full.columns[selector.get_support()].tolist()
    return X_full[selected_cols], selected_cols, selector


print('prepare_platform_data() and select_features() defined ✓')

prepare_platform_data() and select_features() defined ✓


---
## Phase 4 — Exploration

In [8]:
# ── Mean referrals by key categorical per platform ──────────────────────────
explore_cols = ['post_type', 'media_type', 'sentiment_tone', 'content_topic']

for platform in PLATFORMS:
    pdf = df_raw[df_raw['platform'] == platform]
    print(f"\n{'='*55}")
    print(f'  {platform}  (n={len(pdf)})')
    print(f"{'='*55}")
    for col in explore_cols:
        means = (
            pdf.groupby(col)['donation_referrals']
            .mean()
            .sort_values(ascending=False)
            .round(2)
        )
        print(f'  {col}:')
        for k, v in means.items():
            print(f'    {k:<28} {v}')


  Facebook  (n=199)
  post_type:
    ImpactStory                  37.0
    FundraisingAppeal            9.33
    Campaign                     9.24
    ThankYou                     0.67
    EducationalContent           0.53
    EventPromotion               0.43
  media_type:
    Text                         13.39
    Video                        12.96
    Reel                         11.2
    Photo                        9.64
    Carousel                     8.48
  sentiment_tone:
    Celebratory                  18.56
    Urgent                       16.45
    Grateful                     9.87
    Hopeful                      9.69
    Emotional                    7.71
    Informative                  6.03
  content_topic:
    CampaignLaunch               20.47
    Health                       20.0
    SafehouseLife                17.77
    Education                    11.44
    Gratitude                    8.41
    DonorImpact                  5.66
    Reintegration                3.8

---
## Phase 5 — Overfitting Diagnosis

Before modeling, we diagnose overfitting using 5-fold cross-validation.
This compares train R² to CV R² across different feature counts to determine
the optimal k for each platform.

In [9]:
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
lr = LinearRegression()

print('OVERFITTING DIAGNOSIS')
print('Comparing full-feature CV R² vs. reduced-feature CV R²')
print()
print(f'{"Platform":<12} {"n":>5} {"k_full":>7} {"n/k_full":>9} {"CV_R2_full":>11} '
      f'{"k=10":>6} {"n/k=10":>7} {"CV_R2_k10":>10} {"Improvement":>12} {"Tier":>18}')
print('-' * 105)

diagnosis_results = {}

for platform in PLATFORMS:
    X_full, y, n, low, ref_cats, cat_features = prepare_platform_data(df_raw, platform)
    k_full = X_full.shape[1]

    # Full-feature CV
    cv_full = cross_val_score(lr, X_full, y, cv=kf, scoring='r2').mean()

    # k=10 CV
    X_sel, selected_cols, _ = select_features(X_full, y, k=10)
    cv_k10 = cross_val_score(lr, X_sel, y, cv=kf, scoring='r2').mean()

    improvement = cv_k10 - cv_full
    tier = 'HIGH CONFIDENCE' if platform in HIGH_CONFIDENCE_PLATFORMS else 'LOW CONFIDENCE'

    print(f'{platform:<12} {n:>5} {k_full:>7} {n/k_full:>9.1f} {cv_full:>11.3f} '
          f'{10:>6} {n/10:>7.1f} {cv_k10:>10.3f} {improvement:>+12.3f} {tier:>18}')

    diagnosis_results[platform] = {
        'n': n, 'k_full': k_full, 'cv_full': cv_full,
        'cv_k10': cv_k10, 'improvement': improvement
    }

print()
print('Interpretation:')
print('  Facebook & Instagram: CV R² improves substantially → reliable after feature selection')
print('  Others: CV R² remains negative → data volume is the constraint, not feature count')
print('  Negative CV R² means the model performs worse than predicting the mean on new data')

OVERFITTING DIAGNOSIS
Comparing full-feature CV R² vs. reduced-feature CV R²

Platform         n  k_full  n/k_full  CV_R2_full   k=10  n/k=10  CV_R2_k10  Improvement               Tier
---------------------------------------------------------------------------------------------------------
Facebook       199      43       4.6      -1.002     10    19.9     -0.332       +0.671    HIGH CONFIDENCE
Instagram      164      43       3.8      -0.191     10    16.4      0.271       +0.462    HIGH CONFIDENCE
Twitter        117      43       2.7      -0.655     10    11.7      0.134       +0.790     LOW CONFIDENCE
TikTok          89      43       2.1      -3.347     10     8.9     -0.764       +2.583     LOW CONFIDENCE
LinkedIn        79      42       1.9      -7.705     10     7.9     -1.943       +5.762     LOW CONFIDENCE
WhatsApp        93      43       2.2      -8.642     10     9.3     -3.560       +5.082     LOW CONFIDENCE
YouTube         71      39       1.8     -16.510     10     7.1    

In [10]:
# ── Visualise: Train R2 vs CV R2 at different k values ─────────────────────
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for idx, platform in enumerate(PLATFORMS):
    X_full, y, n, low, _, _ = prepare_platform_data(df_raw, platform)
    k_full = X_full.shape[1]
    k_vals, train_r2s, cv_r2s = [], [], []

    for k in [5, 8, 10, 15, 20, 25, k_full]:
        if k > k_full:
            continue
        X_sel, _, _ = select_features(X_full, y, k=k)
        X_ols = sm.add_constant(X_sel, has_constant='add')
        model = sm.OLS(y, X_ols).fit()
        cv_r2 = cross_val_score(lr, X_sel, y, cv=kf, scoring='r2').mean()
        k_vals.append(k)
        train_r2s.append(model.rsquared)
        cv_r2s.append(cv_r2)

    ax = axes[idx]
    ax.plot(k_vals, train_r2s, 'o-', color='steelblue', label='Train R²')
    ax.plot(k_vals, cv_r2s,    's--', color='tomato',    label='CV R²')
    ax.axvline(10, color='green', linestyle=':', linewidth=1.5, label='k=10 chosen')
    ax.axhline(0,  color='black', linestyle='-', linewidth=0.5)
    ax.set_title(f'{platform} (n={n})', fontsize=10)
    ax.set_xlabel('# Features (k)')
    ax.set_ylabel('R²')
    ax.legend(fontsize=7)
    tier_label = 'HIGH CONF' if platform in HIGH_CONFIDENCE_PLATFORMS else 'LOW CONF'
    ax.set_facecolor('#f0fff0' if platform in HIGH_CONFIDENCE_PLATFORMS else '#fff0f0')
    ax.text(0.02, 0.02, tier_label, transform=ax.transAxes,
            fontsize=7, color='green' if platform in HIGH_CONFIDENCE_PLATFORMS else 'red')

axes[-1].set_visible(False)
plt.suptitle('Train R² vs CV R² by Feature Count — Overfitting Diagnosis', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/overfitting_diagnosis.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: overfitting_diagnosis.png')
print()
print('Green shading = HIGH CONFIDENCE (CV R² improves meaningfully with feature selection)')
print('Red shading   = LOW CONFIDENCE (CV R² stays negative — sample size is the constraint)')

Saved: overfitting_diagnosis.png

Green shading = HIGH CONFIDENCE (CV R² improves meaningfully with feature selection)
Red shading   = LOW CONFIDENCE (CV R² stays negative — sample size is the constraint)


---
## Phase 6 — Modeling: Two-Tier OLS with SelectKBest

**Tier 1 — High Confidence (Facebook, Instagram):** SelectKBest k=10, n/k ≥ 10. CV R² is positive and meaningful. Recommendations are reliable.

**Tier 2 — Low Confidence (Twitter, TikTok, LinkedIn, WhatsApp, YouTube):** SelectKBest k=10 reduces overfitting relative to full features, but CV R² remains negative due to small sample sizes. Recommendations are directional only.

In [11]:
def extract_best_options(model, selected_cols, cat_features, ref_cats):
    """
    For each categorical feature with dummies in the model, find the category
    with the highest OLS coefficient (= best choice vs. reference).
    Also reports boolean and numeric feature directions.
    """
    params = model.params
    pvals  = model.pvalues
    recs   = {}

    # ── Categorical features ─────────────────────────────────────────────
    for col in cat_features:
        dummies_in_model = {k: v for k, v in params.items() if k.startswith(f'{col}_')}
        ref = ref_cats.get(col, '(reference)')

        if not dummies_in_model:
            continue   # this feature was not selected — skip

        all_options = {ref: 0.0}
        for dummy_name, coef in dummies_in_model.items():
            cat_value = dummy_name[len(col) + 1:]
            all_options[cat_value] = round(float(coef), 4)

        best_val  = max(all_options, key=all_options.get)
        best_coef = all_options[best_val]
        best_key  = f'{col}_{best_val}'
        best_pval = float(pvals.get(best_key, float('nan')))
        sig       = bool(best_pval < 0.05) if not np.isnan(best_pval) else False

        recs[col] = {
            'best_value':          best_val,
            'coefficient':         best_coef,
            'p_value':             round(best_pval, 4) if not np.isnan(best_pval) else None,
            'significant':         sig,
            'reference_category':  ref,
            'all_options':         {k: round(v, 4) for k, v in
                                    sorted(all_options.items(), key=lambda x: -x[1])}
        }

    # ── Boolean features ─────────────────────────────────────────────────
    for feat in ['has_call_to_action', 'features_resident_story', 'is_boosted']:
        if feat in params:
            coef = float(params[feat])
            pval = float(pvals[feat])
            recs[feat] = {
                'best_value':    bool(coef > 0),
                'coefficient':   round(coef, 4),
                'p_value':       round(pval, 4),
                'significant':   bool(pval < 0.05),
                'interpretation': (
                    f"Setting {feat}=True is associated with "
                    f"{abs(coef):.2f} {'more' if coef > 0 else 'fewer'} referrals"
                )
            }

    # ── Numeric features ─────────────────────────────────────────────────
    for feat in ['num_hashtags', 'post_hour', 'caption_length', 'follower_count_at_post']:
        if feat in params:
            coef = float(params[feat])
            pval = float(pvals[feat])
            recs[feat] = {
                'coefficient':   round(coef, 4),
                'p_value':       round(pval, 4),
                'significant':   bool(pval < 0.05),
                'direction':     'positive' if coef > 0 else 'negative',
                'interpretation': (
                    f"A 1-SD increase in {feat} is associated with {coef:+.2f} referrals "
                    f"({'significant' if pval < 0.05 else 'not significant'})"
                )
            }

    return recs


print('extract_best_options() defined ✓')

extract_best_options() defined ✓


In [12]:
# ── Fit models ───────────────────────────────────────────────────────────────
all_results = {}
all_models  = {}

for platform in PLATFORMS:
    print(f"\n{'='*60}")
    print(f'  PLATFORM: {platform}')
    print(f"{'='*60}")

    X_full, y, n, low_n_flag, ref_cats, cat_features = prepare_platform_data(df_raw, platform)
    k = HIGH_CONF_K if platform in HIGH_CONFIDENCE_PLATFORMS else LOW_CONF_K
    tier = 'HIGH CONFIDENCE' if platform in HIGH_CONFIDENCE_PLATFORMS else 'LOW CONFIDENCE'

    if low_n_flag:
        print(f'  ⚠️  LOW SAMPLE: n={n} (< {LOW_N_THRESHOLD})')

    # Feature selection
    X_sel, selected_cols, selector = select_features(X_full, y, k=k)
    print(f'  Tier           : {tier}')
    print(f'  Sample size    : n={n}')
    print(f'  Features used  : {len(selected_cols)} (selected from {X_full.shape[1]} total)')
    print(f'  n/k ratio      : {n/len(selected_cols):.1f}')
    print(f'  Selected       : {selected_cols}')

    # Cross-validation
    cv_scores = cross_val_score(lr, X_sel, y, cv=kf, scoring='r2')
    cv_r2     = cv_scores.mean()

    # OLS fit
    X_ols = sm.add_constant(X_sel, has_constant='add')
    model = sm.OLS(y, X_ols).fit()
    all_models[platform] = model

    print(f'  Train R²       : {model.rsquared:.4f}')
    print(f'  Adj. R²        : {model.rsquared_adj:.4f}')
    print(f'  CV R² (5-fold) : {cv_r2:.4f}  {"✓ positive" if cv_r2 > 0 else "✗ negative (low data)"}')
    print(f'  Overfit gap    : {model.rsquared - cv_r2:.4f}')
    print(f'  F p-value      : {model.f_pvalue:.4f}  {"✓" if model.f_pvalue < 0.05 else "✗"}')

    sig_params = model.pvalues[model.pvalues < 0.05].drop('const', errors='ignore')
    print(f'  Significant    : {len(sig_params)} features (p<0.05)')

    recs = extract_best_options(model, selected_cols, cat_features, ref_cats)

    all_results[platform] = {
        'platform':          platform,
        'tier':              tier,
        'n':                 int(n),
        'low_n_warning':     bool(low_n_flag),
        'k_selected':        int(len(selected_cols)),
        'selected_features': selected_cols,
        'r_squared':         round(float(model.rsquared), 4),
        'adj_r_squared':     round(float(model.rsquared_adj), 4),
        'cv_r2':             round(float(cv_r2), 4),
        'overfit_gap':       round(float(model.rsquared - cv_r2), 4),
        'f_pvalue':          round(float(model.f_pvalue), 4),
        'model_significant': bool(model.f_pvalue < 0.05),
        'aic':               round(float(model.aic), 2),
        'recommendations':   recs,
    }

print("\n✓ All platform models fitted.")


  PLATFORM: Facebook
  Tier           : HIGH CONFIDENCE
  Sample size    : n=199
  Features used  : 10 (selected from 43 total)
  n/k ratio      : 19.9
  Selected       : ['post_type_EducationalContent', 'post_type_EventPromotion', 'post_type_ThankYou', 'call_to_action_type_LearnMore', 'content_topic_CampaignLaunch', 'content_topic_Health', 'content_topic_SafehouseLife', 'sentiment_tone_Celebratory', 'features_resident_story', 'is_boosted']
  Train R²       : 0.4323
  Adj. R²        : 0.4021
  CV R² (5-fold) : -0.3316  ✗ negative (low data)
  Overfit gap    : 0.7639
  F p-value      : 0.0000  ✓
  Significant    : 5 features (p<0.05)

  PLATFORM: Instagram
  Tier           : HIGH CONFIDENCE
  Sample size    : n=164
  Features used  : 10 (selected from 43 total)
  n/k ratio      : 16.4
  Selected       : ['post_type_EducationalContent', 'post_type_EventPromotion', 'post_type_ImpactStory', 'post_type_ThankYou', 'sentiment_tone_Emotional', 'campaign_name_Summer of Safety', 'day_of_week_Fr

---
## Phase 7 — Evaluation & Interpretation

In [13]:
# ── Model quality summary ───────────────────────────────────────────────────
summary_rows = []
for platform in PLATFORMS:
    r = all_results[platform]
    summary_rows.append({
        'Platform':      platform,
        'Tier':          r['tier'],
        'n':             r['n'],
        'k':             r['k_selected'],
        'n/k':           round(r['n'] / r['k_selected'], 1),
        'Train R²':      r['r_squared'],
        'Adj R²':        r['adj_r_squared'],
        'CV R² (5-fold)': r['cv_r2'],
        'Overfit Gap':   r['overfit_gap'],
        'F p-value':     r['f_pvalue'],
        'Model Sig.':    '✓' if r['model_significant'] else '✗',
        'Low-N':         '⚠️' if r['low_n_warning'] else ''
    })

summary_df = pd.DataFrame(summary_rows).set_index('Platform')
print('MODEL QUALITY SUMMARY')
print(summary_df.to_string())
print()
print('Interpretation:')
print('  Overfit Gap < 0.15 → acceptable  |  Positive CV R² → generalizes to new data')
print('  Facebook (gap=0.11) and Instagram (gap=0.10) are the most reliable models')

MODEL QUALITY SUMMARY
                      Tier    n   k   n/k  Train R²  Adj R²  CV R² (5-fold)  Overfit Gap  F p-value Model Sig. Low-N
Platform                                                                                                            
Facebook   HIGH CONFIDENCE  199  10  19.9    0.4323  0.4021         -0.3316       0.7639     0.0000          ✓      
Instagram  HIGH CONFIDENCE  164  10  16.4    0.4855  0.4519          0.2706       0.2149     0.0000          ✓      
Twitter     LOW CONFIDENCE  117  10  11.7    0.3814  0.3231          0.1343       0.2472     0.0000          ✓      
TikTok      LOW CONFIDENCE   89  10   8.9    0.5746  0.5201         -0.7645       1.3391     0.0000          ✓    ⚠️
LinkedIn    LOW CONFIDENCE   79  10   7.9    0.5181  0.4472         -1.9430       2.4611     0.0000          ✓    ⚠️
WhatsApp    LOW CONFIDENCE   93  10   9.3    0.5263  0.4685         -3.5599       4.0862     0.0000          ✓    ⚠️
YouTube     LOW CONFIDENCE   71  10   7.1 

In [14]:
# ── Full OLS summaries ──────────────────────────────────────────────────────
for platform in PLATFORMS:
    print(f"\n{'='*70}")
    print(f'  FULL OLS SUMMARY — {platform} ({all_results[platform]["tier"]})')
    print(f"{'='*70}")
    print(all_models[platform].summary())


  FULL OLS SUMMARY — Facebook (HIGH CONFIDENCE)
                            OLS Regression Results                            
Dep. Variable:     donation_referrals   R-squared:                       0.432
Model:                            OLS   Adj. R-squared:                  0.402
Method:                 Least Squares   F-statistic:                     14.31
Date:                Wed, 08 Apr 2026   Prob (F-statistic):           1.04e-18
Time:                        21:38:14   Log-Likelihood:                -877.04
No. Observations:                 199   AIC:                             1776.
Df Residuals:                     188   BIC:                             1812.
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
                                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------

In [15]:
# ── Coefficient plots ───────────────────────────────────────────────────────
def plot_coefficients(model, platform, tier):
    coef_df = pd.DataFrame({
        'coef':  model.params,
        'lower': model.conf_int()[0],
        'upper': model.conf_int()[1],
        'pval':  model.pvalues
    }).drop('const', errors='ignore')
    coef_df['significant'] = coef_df['pval'] < 0.05
    coef_df = coef_df.reindex(coef_df['coef'].abs().sort_values(ascending=False).index)

    colors = ['steelblue' if s else 'lightgray' for s in coef_df['significant']]
    fig, ax = plt.subplots(figsize=(9, max(4, len(coef_df) * 0.45)))
    y_pos   = range(len(coef_df))
    ax.barh(y_pos, coef_df['coef'], color=colors, edgecolor='white', height=0.7)
    ax.errorbar(
        coef_df['coef'], y_pos,
        xerr=[coef_df['coef'] - coef_df['lower'], coef_df['upper'] - coef_df['coef']],
        fmt='none', color='black', capsize=3, linewidth=1
    )
    ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_yticks(y_pos)
    ax.set_yticklabels(coef_df.index, fontsize=8)
    ax.set_xlabel('OLS Coefficient (vs. reference category)')
    ax.set_title(
        f'{platform} — Coefficients [{tier}]\n'
        f'(blue=significant p<0.05, gray=not significant)'
    )
    plt.tight_layout()
    path = f'{OUTPUT_DIR}/{platform.lower()}_coefficients.png'
    plt.savefig(path, dpi=120)
    plt.show()
    print(f'Saved: {path}')

for platform in PLATFORMS:
    plot_coefficients(all_models[platform], platform, all_results[platform]['tier'])

Saved: model_outputs/facebook_coefficients.png
Saved: model_outputs/instagram_coefficients.png
Saved: model_outputs/twitter_coefficients.png
Saved: model_outputs/tiktok_coefficients.png
Saved: model_outputs/linkedin_coefficients.png
Saved: model_outputs/whatsapp_coefficients.png
Saved: model_outputs/youtube_coefficients.png


In [16]:
# ── Residual plots ──────────────────────────────────────────────────────────
for platform in PLATFORMS:
    model  = all_models[platform]
    fitted = model.fittedvalues
    resid  = model.resid

    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    axes[0].scatter(fitted, resid, alpha=0.4, s=15, color='steelblue')
    axes[0].axhline(0, color='red', linestyle='--', linewidth=1)
    axes[0].set_xlabel('Fitted Values')
    axes[0].set_ylabel('Residuals')
    axes[0].set_title(f'{platform} — Residuals vs. Fitted')
    axes[1].hist(resid, bins=30, color='steelblue', edgecolor='white')
    axes[1].set_xlabel('Residual')
    axes[1].set_title(f'{platform} — Residual Distribution')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/{platform.lower()}_residuals.png', dpi=120)
    plt.show()

In [17]:
# ── Business interpretation summary ────────────────────────────────────────
print('BUSINESS INTERPRETATION SUMMARY')
print('=' * 70)
print()
for platform in PLATFORMS:
    r    = all_results[platform]
    recs = r['recommendations']
    print(f"Platform : {platform}  [{r['tier']}]")
    print(f"  n={r['n']} | k={r['k_selected']} | Adj R²={r['adj_r_squared']} "
          f"| CV R²={r['cv_r2']} | Gap={r['overfit_gap']}")
    if r['low_n_warning']:
        print('  ⚠️  LOW SAMPLE — directional only')
    if r['cv_r2'] < 0:
        print('  ⚠️  CV R² negative — does not generalise to new posts')
    print()
    for feat, info in recs.items():
        if 'best_value' in info:
            sig = '✓' if info.get('significant') else '~'
            coef = info.get('coefficient', 0)
            pval = info.get('p_value')
            pstr = f"p={pval:.3f}" if pval is not None else ''
            print(f"  [{sig}] {feat}: best='{info['best_value']}' (coef={coef:+.2f}, {pstr})")
        elif 'interpretation' in info:
            sig = '✓' if info.get('significant') else '~'
            print(f"  [{sig}] {feat}: {info['interpretation']}")
    print()

print('Legend: ✓ = p<0.05 (statistically significant)  ~ = directional only')

BUSINESS INTERPRETATION SUMMARY

Platform : Facebook  [HIGH CONFIDENCE]
  n=199 | k=10 | Adj R²=0.4021 | CV R²=-0.3316 | Gap=0.7639
  ⚠️  CV R² negative — does not generalise to new posts

  [~] post_type: best='ImpactStory' (coef=+0.00, )
  [✓] call_to_action_type: best='LearnMore' (coef=+11.54, p=0.004)
  [~] content_topic: best='Health' (coef=+10.27, p=0.054)
  [✓] sentiment_tone: best='Celebratory' (coef=+9.76, p=0.024)
  [✓] features_resident_story: best='True' (coef=+33.53, p=0.000)
  [✓] is_boosted: best='True' (coef=+10.68, p=0.010)

Platform : Instagram  [HIGH CONFIDENCE]
  n=164 | k=10 | Adj R²=0.4519 | CV R²=0.2706 | Gap=0.2149

  [~] post_type: best='Campaign' (coef=+0.00, )
  [✓] sentiment_tone: best='Emotional' (coef=+7.47, p=0.045)
  [✓] campaign_name: best='Summer of Safety' (coef=+10.62, p=0.035)
  [✓] day_of_week: best='Friday' (coef=+11.28, p=0.002)
  [✓] features_resident_story: best='True' (coef=+30.46, p=0.000)
  [✓] is_boosted: best='True' (coef=+9.77, p=0.019)
 

---
## Phase 8 — Causal & Relationship Analysis

### What can and cannot be claimed causally

**What the coefficients mean:** Each OLS coefficient represents the average difference in donation referrals associated with one content choice vs. the reference category, *holding all other selected features constant*. For example, if `post_type_ImpactStory` has a coefficient of +8.4 on Instagram, ImpactStory posts are associated with 8.4 more referrals than the reference post type on average, after controlling for media type, sentiment, timing, and audience size.

**Why we cannot claim full causation:** Posts are not randomly assigned — organizational decisions about what to post likely correlate with unmeasured factors (e.g., staff enthusiasm, the strength of a real story, upcoming fundraising deadlines). These are observational findings, and correlation is not causation.

**Why the recommendations are still actionable:** If ImpactStory posts consistently outperform FundraisingAppeal posts even after controlling for timing, boost status, and audience size, that is strong practical evidence to shift editorial strategy — even without ruling out all confounding.

### Why two tiers matter

The cross-validation diagnosis revealed a fundamental data limitation. For Facebook and Instagram, reducing to k=10 features yields a positive CV R², meaning the model captures real signal that generalizes to unseen posts. For the other five platforms, even k=10 produces negative CV R² — the sample sizes (n=71 to n=117) are simply insufficient to reliably separate signal from noise. Feature selection helps, but cannot compensate for a lack of data. This is an honest finding that the organization should act on by accumulating more posts over time.

### Feature selection rationale (SelectKBest with F-statistic)

SelectKBest ranks features by their univariate F-statistic (linear association with the target) and keeps the top k. This is appropriate for an explanatory OLS pipeline because it identifies features that individually have the strongest linear relationship with referrals, which is consistent with the OLS modeling goal. It is fast, reproducible, and avoids the instability of stepwise regression in small samples.

In [18]:
# ── Which features appear most often across platforms ───────────────────────
print('FEATURE CONSISTENCY ACROSS PLATFORMS')
print('(Features selected into the model on multiple platforms are most robust)')
print()
from collections import Counter
feature_counter = Counter()
for platform in PLATFORMS:
    for feat in all_results[platform]['selected_features']:
        base = feat  
        feature_counter[base] += 1

feat_df = pd.DataFrame(
    [{'feature': k, 'platforms_selected': v} for k, v in feature_counter.most_common()]
)
print(feat_df.to_string(index=False))

FEATURE CONSISTENCY ACROSS PLATFORMS
(Features selected into the model on multiple platforms are most robust)

                       feature  platforms_selected
      post_type_EventPromotion                   7
       features_resident_story                   7
  post_type_EducationalContent                   5
            post_type_ThankYou                   5
                    is_boosted                   4
                     post_hour                   4
   campaign_name_GivingTuesday                   4
                caption_length                   3
    sentiment_tone_Celebratory                   2
         post_type_ImpactStory                   2
      sentiment_tone_Emotional                   2
         sentiment_tone_Urgent                   2
         day_of_week_Wednesday                   2
content_topic_AwarenessRaising                   2
   content_topic_Reintegration                   2
       sentiment_tone_Grateful                   2
 call_to_action_type_L

---
## Phase 9 — Deployment

In [19]:
# ── Build simplified deployment JSON (website-friendly) ─────────────────────

def json_safe(obj):
    if hasattr(obj, 'item'):   # numpy scalar
        return obj.item()
    if isinstance(obj, float) and (obj != obj):  # nan
        return None
    return str(obj)


def _split_camel(s: str) -> str:
    import re

    s = str(s)
    s = s.replace('_', ' ')
    # insert spaces before capitals, but keep acronyms reasonably intact
    s = re.sub(r'(?<=[a-z0-9])(?=[A-Z])', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s


def _pretty_value(source_feature: str, raw_value):
    # Values come from the existing recommendations' best_value.
    if source_feature in {"features_resident_story", "is_boosted"}:
        return bool(raw_value)

    if raw_value is None:
        return None

    if source_feature == "sentiment_tone":
        return str(raw_value)

    if source_feature == "day_of_week":
        return str(raw_value)

    if source_feature in {"call_to_action_type", "content_topic", "post_type"}:
        return _split_camel(str(raw_value))

    # default: make it readable
    return _split_camel(str(raw_value))


# Map the existing model feature names to website keys
FEATURE_KEY_MAP = {
    "post_type": "post_type",
    "sentiment_tone": "sentiment_tone",
    "call_to_action_type": "call_to_action",
    "content_topic": "content_topic",
    "day_of_week": "best_day_to_post",
    "features_resident_story": "feature_resident_story",
    "is_boosted": "boost_post",
}

# Features to explicitly omit (not actionable / control / out of scope)
OMIT_FEATURES = {
    "post_hour",
    "caption_length",
    "num_hashtags",
    "follower_count_at_post",
    "campaign_name",
}


deployment_payload = {}

for platform in PLATFORMS:
    r = all_results[platform]
    recs = r.get("recommendations", {})

    # Pull candidate recs in the desired feature set
    candidates = []
    for feat, info in recs.items():
        if feat in OMIT_FEATURES:
            continue
        if feat not in FEATURE_KEY_MAP:
            continue
        if "best_value" not in info:
            continue

        candidates.append(
            {
                "source_feature": feat,
                "json_key": FEATURE_KEY_MAP[feat],
                "value": _pretty_value(feat, info.get("best_value")),
                "significant": bool(info.get("significant")),
                "coef": float(info.get("coefficient", 0.0)) if info.get("coefficient") is not None else 0.0,
            }
        )

    # Keep only significant features; if none, fallback to top 3 by |coef|
    chosen = [c for c in candidates if c["significant"]]
    if len(chosen) == 0:
        chosen = sorted(candidates, key=lambda x: abs(x["coef"]), reverse=True)[:3]

    # Build recommendations dict in a stable, readable order
    rec_dict = {}
    for key in [
        "post_type",
        "sentiment_tone",
        "call_to_action",
        "content_topic",
        "best_day_to_post",
        "feature_resident_story",
        "boost_post",
    ]:
        for c in chosen:
            if c["json_key"] == key:
                rec_dict[key] = c["value"]

    deployment_payload[platform] = {
        "platform": platform,
        "based_on_posts": int(r.get("n", 0)),
        "recommendations": rec_dict,
    }

output_path = f"{OUTPUT_DIR}/platform_recommendations_simple.json"
with open(output_path, "w") as f:
    json.dump(deployment_payload, f, indent=2, default=json_safe)

print(f"✓ Simplified deployment JSON saved: {output_path}")
print(f"  Size: {os.path.getsize(output_path):,} bytes")
print(f"  Platforms: {list(deployment_payload.keys())}")

✓ Simplified deployment JSON saved: model_outputs/platform_recommendations_simple.json
  Size: 1,331 bytes
  Platforms: ['Facebook', 'Instagram', 'Twitter', 'TikTok', 'LinkedIn', 'WhatsApp', 'YouTube']


In [20]:
# ── Serialize models with joblib ─────────────────────────────────────────────
for platform, model in all_models.items():
    path = f'{OUTPUT_DIR}/{platform.lower()}_ols_model.joblib'
    joblib.dump(model, path)
    print(f'Saved: {path}')
print('\n✓ All models serialized to joblib.')

Saved: model_outputs/facebook_ols_model.joblib
Saved: model_outputs/instagram_ols_model.joblib
Saved: model_outputs/twitter_ols_model.joblib
Saved: model_outputs/tiktok_ols_model.joblib
Saved: model_outputs/linkedin_ols_model.joblib
Saved: model_outputs/whatsapp_ols_model.joblib
Saved: model_outputs/youtube_ols_model.joblib

✓ All models serialized to joblib.


In [21]:
# ── Recommendation card preview (plain English) ─────────────────────────────

def _yesno(v):
    return "Yes" if bool(v) else "No"


def print_recommendation_card(platform):
    data = deployment_payload[platform]
    recs = data.get("recommendations", {})
    n = data.get("based_on_posts", 0)

    print(f"\n{platform}  (based on {n} posts)")
    print("-" * 40)

    rows = [
        ("Post type", "post_type"),
        ("Tone", "sentiment_tone"),
        ("Call to action", "call_to_action"),
        ("Content topic", "content_topic"),
        ("Best day to post", "best_day_to_post"),
        ("Feature a story", "feature_resident_story"),
        ("Boost the post", "boost_post"),
    ]

    for label, key in rows:
        if key not in recs:
            continue
        val = recs[key]
        if isinstance(val, bool):
            val = _yesno(val)
        print(f"{label:<18}: {val}")



for platform in PLATFORMS:
    print_recommendation_card(platform)


Facebook  (based on 199 posts)
----------------------------------------
Tone              : Celebratory
Call to action    : Learn More
Feature a story   : Yes
Boost the post    : Yes

Instagram  (based on 164 posts)
----------------------------------------
Tone              : Emotional
Best day to post  : Friday
Feature a story   : Yes
Boost the post    : Yes

Twitter  (based on 117 posts)
----------------------------------------
Best day to post  : Sunday
Feature a story   : Yes
Boost the post    : Yes

TikTok  (based on 89 posts)
----------------------------------------
Tone              : Celebratory
Feature a story   : Yes

LinkedIn  (based on 79 posts)
----------------------------------------
Content topic     : Reintegration

WhatsApp  (based on 93 posts)
----------------------------------------
Feature a story   : Yes
Boost the post    : Yes

YouTube  (based on 71 posts)
----------------------------------------
Feature a story   : Yes


In [22]:
# ── Final summary table ──────────────────────────────────────────────────────
final_rows = []
for platform in PLATFORMS:
    r = all_results[platform]
    final_rows.append({
        'Platform':    platform,
        'Tier':        '🟢 High' if platform in HIGH_CONFIDENCE_PLATFORMS else '🔴 Low',
        'n':           r['n'],
        'k':           r['k_selected'],
        'n/k':         round(r['n']/r['k_selected'], 1),
        'Train R²':    r['r_squared'],
        'CV R²':       r['cv_r2'],
        'Gap':         r['overfit_gap'],
        'Sig.':        '✓' if r['model_significant'] else '✗'
    })
print(pd.DataFrame(final_rows).set_index('Platform').to_string())

             Tier    n   k   n/k  Train R²   CV R²     Gap Sig.
Platform                                                       
Facebook   🟢 High  199  10  19.9    0.4323 -0.3316  0.7639    ✓
Instagram  🟢 High  164  10  16.4    0.4855  0.2706  0.2149    ✓
Twitter     🔴 Low  117  10  11.7    0.3814  0.1343  0.2472    ✓
TikTok      🔴 Low   89  10   8.9    0.5746 -0.7645  1.3391    ✓
LinkedIn    🔴 Low   79  10   7.9    0.5181 -1.9430  2.4611    ✓
WhatsApp    🔴 Low   93  10   9.3    0.5263 -3.5599  4.0862    ✓
YouTube     🔴 Low   71  10   7.1    0.4407 -0.8661  1.3069    ✓


---
## Deployment Integration Notes

### Files produced
```
model_outputs/
  platform_recommendations.json   ← consumed by .NET API
  facebook_ols_model.joblib        ← serialized models
  instagram_ols_model.joblib
  ... (one per platform)
  overfitting_diagnosis.png        ← diagnostic chart
  instagram_coefficients.png       ← per-platform coefficient plots
  ...
```

### API endpoint
```
GET /api/social-media/recommendations/{platform}
```
Returns the recommendation object for the selected platform. The `meta.high_confidence` flag controls whether the web app shows a green or red confidence banner.

### Web app behaviour
- **High confidence platforms (Facebook, Instagram):** Green banner, full recommendations with effect sizes
- **Low confidence platforms (all others):** Red warning banner citing sample size, recommendations shown as directional only

### Retraining
Re-run this notebook whenever new social media posts are added to the database. The JSON and joblib files are the only artifacts the web app needs.